# Phase 3 Testing: Advanced Analytics & Dashboards

This notebook validates all Phase 3 analytics modules:
- Population Analytics: Segment analysis and risk profiling
- Recommendation Engine: Model improvement suggestions
- Advanced Validator: Sensitivity and stress testing
- Dashboard Generator: Performance reporting

In [ ]:
import sys
sys.path.insert(0, '../')

import pandas as pd
import numpy as np
from datetime import datetime
import json
import tempfile
import os

# Import Phase 3 modules
from src.account_score.analytics import (
    PopulationAnalytics,
    RecommendationEngine,
    AdvancedValidator,
)
from src.account_score.analytics.dashboard import DashboardGenerator

print("✓ All imports successful")

## Test 1: Population Analytics - Segment Analysis

In [ ]:
# Create sample physician data
np.random.seed(42)

test_data = pd.DataFrame({
    'physician_id': [f'PHY{i:05d}' for i in range(1000)],
    'composite_score': np.random.normal(5.5, 1.5, 1000).clip(1, 10),
    'adequacy_score': np.random.normal(5.5, 1.5, 1000).clip(1, 10),
    'capacity_score': np.random.normal(5.5, 1.5, 1000).clip(1, 10),
    'appetite_score': np.random.normal(5.5, 1.5, 1000).clip(1, 10),
    'environment_score': np.random.normal(5.5, 1.5, 1000).clip(1, 10),
    'loss_amount': np.random.exponential(25000, 1000),
    'specialty': np.random.choice(['Surgery', 'Medicine', 'Pediatrics', 'Psychiatry'], 1000),
    'state': np.random.choice(['CA', 'TX', 'NY', 'FL', 'IL'], 1000),
    'experience_years': np.random.randint(5, 50, 1000),
})

print(f"Sample data shape: {test_data.shape}")
print(f"\nScore distribution:")
print(test_data[['composite_score', 'adequacy_score', 'capacity_score']].describe())

In [ ]:
# Test 1.1: Segment Analysis
analytics = PopulationAnalytics()
segment_analysis = analytics.analyze_by_segment(
    test_data,
    'composite_score',
    ['specialty', 'state']
)

print("=" * 60)
print("TEST 1.1: SEGMENT ANALYSIS")
print("=" * 60)

print(f"\n✓ Analyzed {len(segment_analysis)} segment dimensions")

# Show specialty segment analysis
print(f"\nSpecialty segments:")
for specialty, stats in segment_analysis['specialty'].items():
    print(f"  {specialty:15} - Count: {stats['count']:3d}, Mean: {stats['mean']:5.2f}, Std: {stats['std']:5.2f}")

print(f"\nState segments (sample):")
for i, (state, stats) in enumerate(list(segment_analysis['state'].items())[:3]):
    print(f"  {state:15} - Count: {stats['count']:3d}, Mean: {stats['mean']:5.2f}, Range: [{stats['min']:4.2f}, {stats['max']:4.2f}]")

## Test 2: Population Analytics - Risk Profiling

In [ ]:
# Test 1.2: Risk Profile Analysis
risk_profiles = analytics.risk_profile_comparison(
    test_data,
    'composite_score',
    risk_thresholds={
        'low': (1.0, 4.0),
        'medium': (4.0, 7.0),
        'high': (7.0, 10.0),
    }
)

print("=" * 60)
print("TEST 1.2: RISK PROFILE ANALYSIS")
print("=" * 60)

print(f"\nTotal physicians: {risk_profiles['total_physicians']}")
print(f"\nRisk distribution:")
for risk_level, profile in risk_profiles['risk_profiles'].items():
    print(f"  {risk_level.upper():10} - Count: {profile['count']:4d} ({profile['percentage']:5.1f}%)")

# Verify total adds up
total = sum(p['count'] for p in risk_profiles['risk_profiles'].values())
assert total == risk_profiles['total_physicians'], "Risk profile counts don't match"
print(f"\n✓ Risk profile counts validated (total: {total})")

## Test 3: Population Analytics - Outlier Detection

In [ ]:
# Test 1.3: Outlier Segment Detection
outliers = analytics.identify_outlier_segments(
    test_data,
    'composite_score',
    'specialty',
    zscore_threshold=1.5
)

print("=" * 60)
print("TEST 1.3: OUTLIER SEGMENT DETECTION")
print("=" * 60)

print(f"\nFound {len(outliers)} outlier segments (zscore > 1.5)")
if outliers:
    for outlier in outliers:
        print(f"\n  {outlier['segment_value']:15} (Severity: {outlier['severity']})")
        print(f"    Mean Score: {outlier['mean_score']:.2f} (vs {outlier['overall_mean']:.2f} overall)")
        print(f"    Z-Score: {outlier['zscore']:.2f}")
        print(f"    Sample Size: {outlier['count']}")
else:
    print("\n✓ No significant outlier segments detected")

## Test 4: Recommendation Engine - Weight Adjustments

In [ ]:
# Test 2.1: Weight Adjustment Recommendations
engine = RecommendationEngine()

# Simulate component correlations with loss
correlations = {
    'adequacy': 0.35,      # Strong
    'capacity': 0.12,      # Weak
    'appetite': 0.22,      # Moderate
    'environment': 0.08,   # Very weak
}

baseline_weights = {
    'adequacy': 0.40,
    'capacity': 0.25,
    'appetite': 0.25,
    'environment': 0.10,
}

recommendations = engine.suggest_weight_adjustments(correlations, baseline_weights)

print("=" * 60)
print("TEST 2.1: WEIGHT ADJUSTMENT RECOMMENDATIONS")
print("=" * 60)

print(f"\nComponent correlations to loss:")
for comp, corr in sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True):
    print(f"  {comp:15} - {corr:6.3f}")

print(f"\nGenerated {len(recommendations)} recommendations:")
if recommendations:
    for rec in recommendations:
        print(f"\n  {rec['type'].upper().replace('_', ' ')}")
        print(f"    Component: {rec['component']}")
        print(f"    Current Weight: {rec['current_weight']:.3f}")
        print(f"    Suggested Weight: {rec['suggested_weight']:.3f}")
        print(f"    Reason: {rec['reason']}")
        print(f"    Confidence: {rec['confidence']}")
else:
    print("  (No weight adjustments recommended)")

## Test 5: Recommendation Engine - Threshold Refitting

In [ ]:
# Test 2.2: Threshold Refitting Recommendation
recommendation = engine.suggest_refitting_thresholds(
    test_data,
    'composite_score',
    'loss_amount',
    current_gini=0.12
)

print("=" * 60)
print("TEST 2.2: THRESHOLD REFITTING RECOMMENDATION")
print("=" * 60)

if recommendation:
    print(f"\n✓ Recommendation generated:")
    print(f"  Type: {recommendation['type']}")
    print(f"  Reason: {recommendation['reason']}")
    print(f"  Impact: {recommendation['impact']}")
    print(f"  Timeframe: {recommendation['timeframe']}")
else:
    print(f"\n✓ No refitting needed (discrimination sufficient)")

## Test 6: Advanced Validator - Stress Testing

In [ ]:
# Test 3.1: Stress Testing
validator = AdvancedValidator()

stress_results = validator.stress_test(
    test_data,
    'composite_score',
    stress_levels=[0.5, 0.75, 1.0, 1.5, 2.0]
)

print("=" * 60)
print("TEST 3.1: STRESS TESTING")
print("=" * 60)

print(f"\nBaseline score statistics:")
baseline = test_data['composite_score'].dropna()
print(f"  Mean: {baseline.mean():.2f}, Std: {baseline.std():.2f}")
print(f"  Range: [{baseline.min():.2f}, {baseline.max():.2f}]")

print(f"\nStress test results:")
print(f"{'Stress Level':<15} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'Out of Range':>12}")
print("-" * 60)
for stress_level in ['0.5x', '0.75x', '1.0x', '1.5x', '2.0x']:
    if stress_level in stress_results:
        r = stress_results[stress_level]
        print(f"{stress_level:<15} {r['mean']:8.2f} {r['std']:8.2f} {r['min']:8.2f} {r['max']:8.2f} {r['out_of_range_pct']:11.1f}%")

print(f"\n✓ Stress testing completed successfully")

## Test 7: Advanced Validator - Threshold Impact Analysis

In [ ]:
# Test 3.2: Threshold Impact Analysis
thresholds = {
    'low_risk_cutoff': 4.0,
    'high_risk_cutoff': 7.0,
    'optimal_target': 5.5,
}

threshold_analysis = validator.threshold_impact_analysis(
    test_data,
    'composite_score',
    thresholds
)

print("=" * 60)
print("TEST 3.2: THRESHOLD IMPACT ANALYSIS")
print("=" * 60)

print(f"\nThreshold impact results:")
for threshold_name, analysis in threshold_analysis.items():
    print(f"\n  {threshold_name.upper().replace('_', ' ')}:")
    print(f"    Value: {analysis['threshold']:.2f}")
    print(f"    Above: {analysis['count_above']:4d} ({analysis['pct_above']:5.1f}%)")
    print(f"    Below: {analysis['count_below']:4d} ({analysis['pct_below']:5.1f}%)")

print(f"\n✓ Threshold analysis completed successfully")

## Test 8: Dashboard Generator - Performance Dashboard

In [ ]:
# Test 4.1: Performance Dashboard Generation
generator = DashboardGenerator()

dashboard = generator.generate_performance_dashboard(
    test_data,
    'composite_score',
    'loss_amount',
    ['adequacy_score', 'capacity_score', 'appetite_score', 'environment_score']
)

print("=" * 60)
print("TEST 4.1: PERFORMANCE DASHBOARD GENERATION")
print("=" * 60)

print(f"\nDashboard metadata:")
print(f"  Generated: {dashboard['generated_at']}")
print(f"  Total records: {dashboard['total_records']}")
print(f"  Valid records: {dashboard['valid_records']}")

print(f"\nScore distribution:")
dist = dashboard['score_distribution']
print(f"  Mean: {dist['mean']:.2f}")
print(f"  Median: {dist['median']:.2f}")
print(f"  Std Dev: {dist['std']:.2f}")
print(f"  Range: [{dist['min']:.2f}, {dist['max']:.2f}]")
print(f"  Deciles: P10={dist['p10']:.2f}, P25={dist['p25']:.2f}, P75={dist['p75']:.2f}, P90={dist['p90']:.2f}")

if 'discrimination' in dashboard:
    disc = dashboard['discrimination']
    print(f"\nDiscrimination (Spearman correlation to loss):")
    print(f"  Correlation: {disc['correlation']:.4f}")
    print(f"  P-value: {disc['pvalue']:.6f}")
    print(f"  Significant: {disc['significant']}")

print(f"\nComponent scores:")
for comp_name, comp_stats in dashboard['components'].items():
    print(f"  {comp_name:20} - Mean: {comp_stats['mean']:5.2f}, Std: {comp_stats['std']:5.2f}, Count: {comp_stats['count']}")

print(f"\nRisk distribution:")
risk = dashboard['risk_distribution']
print(f"  Low (1-4):    {risk['low']:4d} ({risk['low']/dashboard['total_records']*100:5.1f}%)")
print(f"  Medium (4-7): {risk['medium']:4d} ({risk['medium']/dashboard['total_records']*100:5.1f}%)")
print(f"  High (7-10):  {risk['high']:4d} ({risk['high']/dashboard['total_records']*100:5.1f}%)")

print(f"\n✓ Performance dashboard generated successfully")

## Test 9: Dashboard Generator - HTML & JSON Export

In [ ]:
# Test 4.2: Dashboard Export (HTML and JSON)
print("=" * 60)
print("TEST 4.2: DASHBOARD EXPORT")
print("=" * 60)

with tempfile.TemporaryDirectory() as tmpdir:
    # Export as JSON
    json_path = f"{tmpdir}/dashboard.json"
    generator.export_dashboard_json(dashboard, json_path)
    
    assert os.path.exists(json_path), "JSON file not created"
    json_size = os.path.getsize(json_path)
    print(f"\n✓ JSON export successful")
    print(f"  File: {json_path}")
    print(f"  Size: {json_size} bytes")
    
    # Verify JSON is valid
    with open(json_path, 'r') as f:
        json_data = json.load(f)
    print(f"  Keys: {list(json_data.keys())}")
    
    # Export as HTML
    html_path = f"{tmpdir}/dashboard.html"
    generator.export_dashboard_html(dashboard, html_path)
    
    assert os.path.exists(html_path), "HTML file not created"
    html_size = os.path.getsize(html_path)
    print(f"\n✓ HTML export successful")
    print(f"  File: {html_path}")
    print(f"  Size: {html_size} bytes")
    
    # Verify HTML content
    with open(html_path, 'r') as f:
        html_content = f.read()
    assert '<html>' in html_content, "HTML not valid"
    assert 'PAS Performance Dashboard' in html_content, "Dashboard title missing"
    print(f"  Verified HTML structure")

## Test 10: Dashboard Generator - Model Comparison

In [ ]:
# Test 4.3: Model Comparison Dashboard
print("=" * 60)
print("TEST 4.3: MODEL COMPARISON DASHBOARD")
print("=" * 60)

# Create comparison data for 3 model versions
model_results = {
    'v1.0.0-baseline': {
        'discrimination': {'correlation': 0.28, 'pvalue': 0.001},
        'gini': 0.24,
        'mean_score': 5.2,
    },
    'v1.1.0-optimized': {
        'discrimination': {'correlation': 0.35, 'pvalue': 0.001},
        'gini': 0.32,
        'mean_score': 5.5,
    },
    'v1.2.0-production': {
        'discrimination': {'correlation': 0.38, 'pvalue': 0.001},
        'gini': 0.34,
        'mean_score': 5.4,
    },
}

comparison = generator.generate_model_comparison_dashboard(model_results)

print(f"\nModel rankings (by discrimination):")
for i, ranked in enumerate(comparison['comparison'], 1):
    print(f"  {i}. {ranked['model']:20} - Discrimination: {ranked['discrimination_score']:6.2f}")

print(f"\n✓ Model comparison dashboard generated successfully")

## Test 11: Integration Test - Complete Analytics Pipeline

In [ ]:
# Test 5: Integration - Full analytics pipeline
print("=" * 60)
print("TEST 5: COMPLETE ANALYTICS PIPELINE INTEGRATION")
print("=" * 60)

# Run complete analytics workflow
print(f"\n1. Population Analytics:")
analytics = PopulationAnalytics()
segments = analytics.analyze_by_segment(test_data, 'composite_score', ['specialty'])
risk = analytics.risk_profile_comparison(test_data, 'composite_score')
print(f"   ✓ Analyzed {len(segments)} segment dimensions")
print(f"   ✓ Generated risk profiles for {risk['total_physicians']} physicians")

print(f"\n2. Recommendation Engine:")
engine = RecommendationEngine()
rec_weights = engine.suggest_weight_adjustments(correlations, baseline_weights)
rec_threshold = engine.suggest_refitting_thresholds(test_data, 'composite_score', 'loss_amount', 0.12)
print(f"   ✓ Generated {len(rec_weights)} weight recommendations")
print(f"   ✓ Evaluated threshold refitting")

print(f"\n3. Advanced Validator:")
validator = AdvancedValidator()
stress = validator.stress_test(test_data, 'composite_score')
threshold_impact = validator.threshold_impact_analysis(test_data, 'composite_score', {'cutoff': 5.5})
print(f"   ✓ Completed stress testing across {len(stress)} scenarios")
print(f"   ✓ Analyzed threshold impacts")

print(f"\n4. Dashboard Generator:")
generator = DashboardGenerator()
dash = generator.generate_performance_dashboard(test_data, 'composite_score', 'loss_amount', [])
print(f"   ✓ Generated performance dashboard")
print(f"   ✓ Dashboard contains {len(dash)} metrics")

print(f"\n" + "=" * 60)
print(f"✓ ALL PHASE 3 TESTS PASSED")
print(f"=" * 60)

## Summary

Phase 3 Testing Summary:

✓ **Test 1**: Population Analytics - Segment Analysis
- Analyzed by specialty and state
- Generated statistics (mean, median, std, quartiles) for each segment

✓ **Test 2**: Population Analytics - Risk Profiling
- Categorized 1000 physicians into low/medium/high risk
- Verified counts and percentages

✓ **Test 3**: Population Analytics - Outlier Detection
- Identified segments with unusual score distributions
- Used Z-score method for anomaly detection

✓ **Test 4**: Recommendation Engine - Weight Adjustments
- Generated recommendations for component weight changes
- Based on correlation analysis with loss outcomes

✓ **Test 5**: Recommendation Engine - Threshold Refitting
- Evaluated need for binning threshold adjustments
- Used discrimination metrics (Gini, correlation)

✓ **Test 6**: Advanced Validator - Stress Testing
- Tested scores under 5 stress levels (0.5x to 2.0x)
- Monitored out-of-range violations at each level

✓ **Test 7**: Advanced Validator - Threshold Impact Analysis
- Analyzed distribution of physicians above/below thresholds
- Useful for setting clinical decision cutoffs

✓ **Test 8**: Dashboard Generator - Performance Dashboard
- Generated comprehensive performance report
- Included score distribution, discrimination, components, risk distribution

✓ **Test 9**: Dashboard Generator - HTML & JSON Export
- Exported dashboard to JSON format
- Exported dashboard to HTML format with styling

✓ **Test 10**: Dashboard Generator - Model Comparison
- Compared multiple model versions
- Ranked by discrimination power

✓ **Test 11**: Integration Test - Complete Analytics Pipeline
- Verified all modules work together
- Validated end-to-end analytics workflow